# Churn Predictor + Retention Advisor — working notebook

Prototype here, then move settled logic into `src/`.

**Milestones:** M1 load+EDA · M2 model+eval · M3 SHAP · M4 grounded LLM advisor → then M5 Streamlit (`src/app.py`), M6 deploy.

**Before running:** `pip install -r ../requirements.txt`, put the dataset in `../data/`, and copy `../.env.example` → `../.env` with your `ANTHROPIC_API_KEY`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, RocCurveDisplay
from sklearn.calibration import CalibrationDisplay

pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42
DATA = '../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'

## M1 — Load & clean

Telco quirks: `TotalCharges` is stored as text and has blanks for brand-new customers; `customerID` is an identifier (drop it); target `Churn` is Yes/No.

In [ ]:
df = pd.read_csv(DATA)
print(df.shape)

# TotalCharges: coerce to numeric (blanks -> NaN), then fill (new customers, tenure 0)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

df = df.drop(columns=['customerID'])
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

df.head()

## EDA — a quick look

Churn is imbalanced (~26%). Note tenure, Contract and MonthlyCharges — usually the strongest churn signals.

In [ ]:
print('Churn rate: {:.1%}'.format(df['Churn'].mean()))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
df.groupby('Contract')['Churn'].mean().plot.bar(ax=ax[0], title='Churn rate by contract', rot=0)
df['tenure'].plot.hist(ax=ax[1], bins=30, title='Tenure (months)')
plt.tight_layout(); plt.show()

## M1–M2 — Encode, split

Keep a readable copy of each customer's original profile (for the LLM later) alongside the encoded matrix the model uses.

In [ ]:
y = df['Churn']
X_raw = df.drop(columns=['Churn'])

# one-hot encode categoricals; keep column names (needed for SHAP + LLM grounding)
X = pd.get_dummies(X_raw, drop_first=True)

X_train, X_test, y_train, y_test, raw_train, raw_test = train_test_split(
    X, y, X_raw, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)

### Baseline (logistic regression) — a score to beat

In [ ]:
base = LogisticRegression(max_iter=1000, class_weight='balanced')
base.fit(X_train, y_train)
base_auc = roc_auc_score(y_test, base.predict_proba(X_test)[:, 1])
print('Baseline ROC-AUC: {:.3f}'.format(base_auc))

## M2 — Gradient-boosted model + honest evaluation

Using sklearn `HistGradientBoostingClassifier` (no extra install). LightGBM is a drop-in alternative if you prefer (it's in requirements; on macOS may need `brew install libomp`).

We care about **ROC-AUC, PR-AUC, recall on the top-risk decile, and calibration** — calibration matters because we multiply the probability by £ value later.

In [ ]:
model = HistGradientBoostingClassifier(
    learning_rate=0.05, max_iter=400, max_depth=None,
    l2_regularization=1.0, class_weight='balanced', random_state=RANDOM_STATE)
model.fit(X_train, y_train)

proba = model.predict_proba(X_test)[:, 1]
print('ROC-AUC: {:.3f}'.format(roc_auc_score(y_test, proba)))
print('PR-AUC : {:.3f}'.format(average_precision_score(y_test, proba)))

# recall on the top-decile of predicted risk
k = int(len(proba) * 0.10)
top_idx = np.argsort(proba)[-k:]
print('Recall @ top 10%: {:.1%}'.format(y_test.iloc[top_idx].sum() / y_test.sum()))

print(classification_report(y_test, (proba >= 0.5).astype(int)))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
RocCurveDisplay.from_predictions(y_test, proba, ax=ax[0]); ax[0].set_title('ROC')
CalibrationDisplay.from_predictions(y_test, proba, n_bins=10, ax=ax[1]); ax[1].set_title('Calibration')
plt.tight_layout(); plt.show()

## M3 — Per-customer explanations (SHAP)

These drivers are the **grounding** for the LLM: the exact features pushing a given customer's risk up.

In [ ]:
import shap

explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_test)
shap.plots.beeswarm(shap_values, max_display=12)

In [ ]:
def top_drivers(i, n=5):
    """Top n features pushing customer i's churn risk UP (positive SHAP)."""
    vals = shap_values.values[i]
    order = np.argsort(vals)[::-1][:n]
    return [(X_test.columns[j], float(vals[j])) for j in order if vals[j] > 0]

# pick the highest-risk customer in the test set as our demo
demo_i = int(np.argmax(proba))
print('Demo customer churn probability: {:.0%}'.format(proba[demo_i]))
print('Top drivers:', top_drivers(demo_i))
raw_test.iloc[demo_i]

## M4 — Grounded LLM retention advisor

Feed the customer's real profile + churn probability + £-at-risk + SHAP drivers to Claude, and ask for a **structured** retention play. The rules: ground every action in the supplied drivers, no invented data.

*(When you build this for real, the `claude-api` skill will harden it — tool-based structured output, prompt caching, retries.)*

In [ ]:
import os, json
from dotenv import load_dotenv
import anthropic

load_dotenv('../.env')
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

SYSTEM = (
    'You are a SaaS Customer Success strategist. Given a customer profile, their churn '
    'probability, the revenue at risk, and the model drivers (the factors raising their '
    'risk), produce a concise retention play. Ground EVERY recommendation in the supplied '
    'drivers — never invent data. Reply with ONLY valid JSON in this shape: '
    '{"risk_summary": str, "drivers_plain": [str], '
    '"retention_actions": [{"action": str, "why": str, "offer": str}], '
    '"suggested_message": str, "priority": "high"|"medium"|"low"}'
)

def retention_play(i, model_name='claude-sonnet-4-6'):
    profile = raw_test.iloc[i].to_dict()
    p = float(proba[i])
    at_risk = p * float(profile.get('MonthlyCharges', 0)) * 12  # rough annual £-at-risk
    drivers = top_drivers(i)
    user = (
        f'Customer profile: {json.dumps(profile)}\n'
        f'Churn probability: {p:.0%}\n'
        f'Approx revenue at risk (annual): £{at_risk:,.0f}\n'
        f'Model drivers raising risk (feature, SHAP weight): {drivers}'
    )
    resp = client.messages.create(
        model=model_name, max_tokens=900,
        system=SYSTEM,
        messages=[{'role': 'user', 'content': user}],
    )
    return json.loads(resp.content[0].text)

play = retention_play(demo_i)
play

In [ ]:
# Grounding guardrail: every action should connect to a real driver.
driver_terms = ' '.join(f.lower().replace('_', ' ') for f, _ in top_drivers(demo_i))
for a in play['retention_actions']:
    grounded = any(w in driver_terms for w in a['why'].lower().split())
    print(('✓' if grounded else '⚠ ungrounded'), '-', a['action'])

## Next
- Move the settled model code → `src/train.py`, SHAP → `src/explain.py`, LLM → `src/llm_advisor.py`.
- **M5**: build the Streamlit app (`src/app.py`) — ranked £-at-risk list, customer detail, generic-vs-grounded toggle.
- **M6**: deploy to Streamlit Community Cloud, write the README, draft the CV bullet.